## Analyze Claims Data 
#### Scott Schmidt
*M.S. Computer Science Student, Illinois State University*

Dataset & Methodology Overview:
* Data Source: weekly_driving_profiles.csv (354 commercial drivers, 8.1M km, 287k trips; April 2021–April 2022).
* Target: Binary indicator for at-fault insurance claims (62 total claims).
* Predictors: Distance, urban driving, speeding, ADAS warnings, and driver/vehicle demographics.
* Modeling: Train/test split evaluated using Logistic Regression and XGBoost.

## Conclusion
Weekly driving data correlates with past claims, but the models cannot predict future risk:
* XGBoost: Highest overall performance (88.75% accuracy, 33% recall).
* Logistic Regression (1:4 weighted): Higher claim detection (37% recall), but lower accuracy (73.38%) and precision (24%).

Limitations: Claims preceded telematics collection, and random splitting caused data leakage across drivers. The data highlights historical patterns, not predictive power. 

In [1]:
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression

file_path = "/kaggle/input/datasets/scottfredschmidt/weekly-driving-profiles-csv/weekly_driving_profiles.csv"
df = pd.read_csv(file_path)
df.head()

,driver_id,week,engine_capacity,road_quality_moderate,slope_flat,motorway,rural,more_than_one_lane,clear_weather,congested,...,harsh_acceleration,harsh_braking,forward_collision,distracted_driver,too_close_distance,lane_departure,driver_making_calls,driver_smoking,fatigue_driving,claims_count
0,0,2021-06-13 00:00:00+00:00,2.299,0.497795,0.661735,0.159487,0.354314,0.223157,0.491282,0.004252,...,0.0,0.0,7.0,212.0,0.0,6.0,0.0,0.0,0.0,0.0
1,0,2021-06-20 00:00:00+00:00,2.299,0.446922,0.682769,0.251670,0.342359,0.291571,0.695668,0.000687,...,0.0,0.0,8.0,593.0,0.0,144.0,2.0,0.0,0.0,0.0
2,0,2021-06-27 00:00:00+00:00,2.299,0.403515,0.792455,0.386323,0.266479,0.477638,0.435115,0.000295,...,0.0,0.0,6.0,75.0,26.0,308.0,3.0,4.0,0.0,0.0
3,0,2021-07-04 00:00:00+00:00,2.299,0.504037,0.972110,0.347816,0.394169,0.521672,1.000000,0.002024,...,0.0,0.0,1.0,1.0,1.0,104.0,8.0,0.0,0.0,0.0
4,0,2021-07-11 00:00:00+00:00,2.299,0.360000,0.962667,0.712000,0.080000,0.829333,1.000000,0.000000,...,0.0,0.0,1.0,0.0,7.0,42.0,0.0,1.0,0.0,0.0


The claim categories were simplified by combining the 2-claim class with the 1-claim class. 
Because the 2-claim observations were limited, merging them into the claim category increased sample size for the positive class and allowed the model to better distinguish between drivers with and without claims.

In [2]:
#print(df['claims_count'].value_counts())
df["claims_count"] = df["claims_count"].replace({2.0: 1.0}) 
print(df['claims_count'].value_counts())

claims_count
0.0    10647
1.0     1881
Name: count, dtype: int64


In [3]:
date_column = "week"  # Change to your actual column name

df[date_column] = pd.to_datetime(
    df[date_column],
    utc=True,
    errors="coerce"
)

# Extract numeric information from the date
df["year"] = df[date_column].dt.year
df["month"] = df[date_column].dt.month
df["day_of_week"] = df[date_column].dt.dayofweek

In [4]:
from sklearn.model_selection import GroupShuffleSplit

groups = df["driver_id"]

X = df.drop(columns=["claims_count", "week", "driver_id"])
y = df["claims_count"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=123
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()
y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()
print(y_train.unique())

[0. 1.]


# Remove the original text/date column
df = df.drop(columns=[date_column, "driver_id"])
print(df['claims_count'].value_counts())

cols = df.drop(columns=["claims_count"])
X = cols
y = df['claims_count']

df.head()

# XGBoost
Without tunning the XGBoost model the accuracy overfits at 0.98643. 
With tunning accuracy goes to 0.8890 which is most likely the more accurate number.

In [5]:
# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=123,
    stratify=y
)

# Create the classifier
model = XGBClassifier()

# Train the model
model.fit(X_train, y_train)

# Predict classes
y_pred = model.predict(X_test)

# Evaluate predictions
accuracy = round(accuracy_score(y_test, y_pred),4)
print("Accuracy:", accuracy)
print(classification_report(y_test, y_pred))

Accuracy: 0.8875
              precision    recall  f1-score   support

         0.0       0.89      0.99      0.94      2130
         1.0       0.81      0.33      0.47       376

    accuracy                           0.89      2506
   macro avg       0.85      0.66      0.70      2506
weighted avg       0.88      0.89      0.87      2506



In [6]:
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit

def gridSearch(): 
    param_grid = {
        "n_estimators":[100],
        "max_depth": [2, 3, 4],
        "learning_rate": [0.03, 0.1],
        "min_child_weight": [1, 5, 10],
        "subsample": [0.7, 0.9],
        "reg_lambda": [1, 5],
    }
    
    search = GridSearchCV(
        XGBClassifier(n_estimators=100, random_state=42),
        param_grid,
        cv=TimeSeriesSplit(n_splits=5),
        scoring="f1_macro",
        n_jobs=-1,
    )
    search.fit(X, y)
    print(search.best_params_, -search.best_score_)
#gridSearch()
# Latest Best: {'learning_rate': 0.03, 'max_depth': 2, 'min_child_weight': 1, 'n_estimators': 100, 'reg_lambda': 5, 'subsample': 0.7} 0.16015325670498087

In [7]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report

# Best parameters from GridSearchCV
best_params = {
    "learning_rate": 0.1,
    "max_depth": 3,
    "min_child_weight": 10,
    "n_estimators": 100,
    "reg_lambda": 5,
    "subsample": 0.9
}

# Create the model using the best parameters
xgb_model = XGBClassifier(
    objective="binary:logistic",
    learning_rate=0.03,
    max_depth=2,
    min_child_weight=1,
    n_estimators=100,
    reg_lambda=5,
    subsample=0.7,
    scale_pos_weight=5.66,
    random_state=42
)
# Train the model
xgb_model.fit(X_train, y_train)

# Return predicted class labels
y_pred = xgb_model.predict(X_test)

# Evaluate predictions
xgb_accuracy = round(accuracy_score(y_test, y_pred), 4)

print("Test accuracy:", xgb_accuracy)
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Test accuracy: 0.5184

Classification Report:
              precision    recall  f1-score   support

         0.0       0.93      0.47      0.62      2130
         1.0       0.21      0.80      0.33       376

    accuracy                           0.52      2506
   macro avg       0.57      0.63      0.48      2506
weighted avg       0.82      0.52      0.58      2506



In [8]:
import shap

from xgboost import XGBClassifier
import shap

# Make sure the column order matches
X_test = X_test[X_train.columns].copy()

xgb_model = XGBClassifier(
    objective="binary:logistic",
    learning_rate=0.03,
    max_depth=2,
    min_child_weight=1,
    n_estimators=100,
    reg_lambda=5,
    subsample=0.7,
    scale_pos_weight=5.66,
    random_state=42
)

# Retrain using the current grouped data
xgb_model.fit(X_train, y_train)

# Recreate the explainer and SHAP values
explainer = shap.Explainer(xgb_model, X_train)
shap_values = explainer(X_test)

print("X_test shape:", X_test.shape)
print("SHAP shape:", shap_values.values.shape)
print("Feature names:", len(shap_values.feature_names))

X_test shape: (2506, 32)
SHAP shape: (2506, 32)
Feature names: 32


# Logistic Regression

In [9]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

weights = [
    None,
    {0: 1, 1: 2},
    {0: 1, 1: 3},
    {0: 1, 1: 4},
    "balanced"
]

for weight in weights:
    log_model = Pipeline([
        ("scaler", StandardScaler()),
        ("logistic", LogisticRegression(
            class_weight=weight,
            max_iter=5000,
            random_state=42
        ))
    ])

    log_model.fit(X_train, y_train)
    predictions = log_model.predict(X_test)

    print("\nClass weight:", weight)
    print("Accuracy:", round(accuracy_score(y_test, predictions), 4))
    print(classification_report(y_test, predictions))


Class weight: None
Accuracy: 0.8476
              precision    recall  f1-score   support

         0.0       0.85      0.99      0.92      2130
         1.0       0.31      0.01      0.03       376

    accuracy                           0.85      2506
   macro avg       0.58      0.50      0.47      2506
weighted avg       0.77      0.85      0.78      2506


Class weight: {0: 1, 1: 2}
Accuracy: 0.8408
              precision    recall  f1-score   support

         0.0       0.85      0.98      0.91      2130
         1.0       0.33      0.06      0.10       376

    accuracy                           0.84      2506
   macro avg       0.59      0.52      0.51      2506
weighted avg       0.78      0.84      0.79      2506


Class weight: {0: 1, 1: 3}
Accuracy: 0.8093
              precision    recall  f1-score   support

         0.0       0.86      0.92      0.89      2130
         1.0       0.29      0.19      0.23       376

    accuracy                           0.81      2506
 

## Results for Logistic Regression
This logistic model provides a recall similiar to XGBoost's best model while having a high accuracy of  0.7338. 

Class weight: {0: 1, 1: 4}
Accuracy: 0.7338
              precision    recall  f1-score   support

         0.0       0.88      0.80      0.84      2130
         1.0       0.24      0.37      0.29       376

    accuracy                           0.73      2506
   macro avg       0.56      0.58      0.57      2506
weighted avg       0.78      0.73      0.75      2506



In [10]:
selected_log_model = Pipeline([
    ("scaler", StandardScaler()),
    ("logistic", LogisticRegression(
        class_weight={0: 1, 1: 4},
        max_iter=5000,
        random_state=42
    ))
])

# Fit the specific selected model
selected_log_model.fit(X_train, y_train)

# Extract the fitted Logistic Regression estimator
logistic_model = selected_log_model.named_steps["logistic"]

# Create coefficient table
final_coefficients_df = (
    pd.DataFrame({
        "Feature": X_train.columns,
        "Coefficient": logistic_model.coef_[0]
    })
    .sort_values(
        "Coefficient",
        key=lambda column: column.abs(),
        ascending=False
    )
    .head(10)
    .reset_index(drop=True)
)

final_coefficients_df["Feature"] = (
    final_coefficients_df["Feature"]
    .str.replace("_", " ")
    .str.title()
)

final_coefficients_df["Coefficient"] = (
    final_coefficients_df["Coefficient"].round(3)
)

final_coefficients_df

,Feature,Coefficient
0,Harsh Acceleration,-2.518
1,Rural,0.866
2,Sum Roundabout,-0.578
3,Motorway,0.554
4,Harsh Braking,0.525
5,Speed Limit Mean,-0.484
6,Engine Capacity,-0.440
7,Sum Yield Sign,0.280
8,Sum Traffic Signal,0.268
9,Total Distance,0.262


In [11]:
print("Done")

Done
